In [1]:
import polars as pl
import numpy as np
from scipy import stats
import altair as alt

In [2]:
df = pl.scan_parquet("data/data_bias.parquet")

In [3]:
def ci_lower(x, confidence = .95):
    x = x.to_numpy()
    n = len(x)

    mean = np.mean(x)
    se = stats.sem(x)
    ci = se * stats.t.pdf((1 + confidence) / 2, n-1)

    return mean - ci

def ci_upper(x, confidence = .95):
    x = x.to_numpy()
    n = len(x)

    mean = np.mean(x)
    se = stats.sem(x)
    ci = se * stats.t.pdf((1 + confidence) / 2, n-1)

    return mean + ci

In [4]:
df.collect()

train acc,train f1,test acc,test f1,adv acc,adv f1,adv distance,depth,bias,distribution,seed,model
f64,f64,f64,f64,f64,f64,f64,i32,f64,str,i32,str
0.830889,0.845105,0.83,0.844037,0.17,0.087912,0.396298,1,3.0,"""moons""",42,"""Tree"""
0.830889,0.845231,0.83,0.842884,0.17,0.095861,0.404744,1,3.0,"""moons""",42,"""Tree"""
0.829333,0.843265,0.848,0.859779,0.152,0.074236,0.403738,1,3.0,"""moons""",42,"""Tree"""
0.830444,0.843999,0.838,0.852995,0.162,0.066815,0.414947,1,3.0,"""moons""",42,"""Tree"""
0.831556,0.845747,0.824,0.838235,0.176,0.096491,0.399182,1,3.0,"""moons""",42,"""Tree"""
…,…,…,…,…,…,…,…,…,…,…,…
0.999444,0.999259,0.594,0.472727,0.53,0.555766,0.190576,24,0.25,"""circles""",179,"""Tree"""
0.998054,0.997401,0.61,0.498715,0.42,0.176136,0.175593,24,0.25,"""circles""",179,"""Tree"""
0.999722,0.999628,0.58,0.487805,0.542,0.646059,0.181045,24,0.25,"""circles""",179,"""Tree"""


In [5]:
cols = ["train acc", "train f1", "test acc", "test f1", 
               "adv acc", "adv f1", "adv distance"]

def agg_metrics(cols):
    res = []
    for col in cols:
        res.extend([
            pl.col(col).mean().alias(f"{col}_mean"),
            pl.col(col).std().alias(f"{col}_std"),
            pl.col(col).map_batches(ci_lower, return_dtype=pl.Float32, returns_scalar=True).alias(f"{col}_ci_lb"),
            pl.col(col).map_batches(ci_upper, return_dtype=pl.Float32, returns_scalar=True).alias(f"{col}_ci_ub"),
        ])
    return res

In [ ]:
df_stats = df.select(pl.exclude("seed")).group_by(
    pl.col("distribution"),
    pl.col("model"),
    pl.col("depth"),
    pl.col("bias")
).agg(
    agg_metrics(cols)
).sort([
    pl.col("distribution"),
    # pl.col("seed"),
    pl.col("depth"),
    pl.col("bias")
])

# df.with_columns(pl.col("train acc").gr)

In [10]:
df_stats.collect()

distribution,model,depth,bias,train acc_mean,train acc_std,train acc_ci_lb,train acc_ci_ub,train f1_mean,train f1_std,train f1_ci_lb,train f1_ci_ub,test acc_mean,test acc_std,test acc_ci_lb,test acc_ci_ub,test f1_mean,test f1_std,test f1_ci_lb,test f1_ci_ub,adv acc_mean,adv acc_std,adv acc_ci_lb,adv acc_ci_ub,adv f1_mean,adv f1_std,adv f1_ci_lb,adv f1_ci_ub,adv distance_mean,adv distance_std,adv distance_ci_lb,adv distance_ci_ub
str,str,i32,f64,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32
"""circles""","""Tree""",1,0.25,0.635027,0.002225,0.634971,0.635084,0.672284,0.001842,0.672236,0.672331,0.506344,0.022327,0.505773,0.506915,0.547262,0.022835,0.546678,0.547846,0.493656,0.022327,0.493085,0.494227,0.442704,0.028774,0.441968,0.44344,0.220577,0.010613,0.220306,0.220849
"""circles""","""Tree""",1,0.5,0.606206,0.001988,0.606157,0.606255,0.679326,0.001389,0.679292,0.67936,0.51988,0.019866,0.51939,0.52037,0.598451,0.017219,0.598027,0.598876,0.48012,0.019866,0.47963,0.48061,0.353199,0.027478,0.352521,0.353877,0.156398,0.008529,0.156188,0.156609
"""circles""","""Tree""",1,0.75,0.588452,0.001587,0.588413,0.588491,0.696458,0.000895,0.696436,0.69648,0.5569,0.014922,0.556532,0.557268,0.668484,0.010651,0.668221,0.668747,0.4431,0.014922,0.442732,0.443468,0.160381,0.023291,0.159806,0.160955,0.142937,0.008849,0.142719,0.143156
"""circles""","""Tree""",1,1.0,0.55541,0.001769,0.555367,0.555454,0.686665,0.000659,0.686649,0.686682,0.54968,0.011792,0.549389,0.549971,0.682173,0.006612,0.68201,0.682336,0.45032,0.011792,0.450029,0.450611,0.057342,0.017013,0.056922,0.057761,0.17766,0.009086,0.177436,0.177885
"""circles""","""Tree""",1,1.25,0.555476,0.002077,0.555424,0.555527,0.685113,0.000734,0.685095,0.685131,0.54928,0.011255,0.549002,0.549558,0.680972,0.007122,0.680796,0.681147,0.45072,0.011255,0.450442,0.450998,0.064385,0.020721,0.063874,0.064896,0.169435,0.009591,0.169198,0.169672
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""moons""","""Tree""",12,2.0,0.999954,0.000094,0.99995,0.999958,0.999952,0.000097,0.999948,0.999957,0.987667,0.004787,0.987454,0.98788,0.987673,0.004779,0.98746,0.987886,0.0984,0.019401,0.097536,0.099264,0.170188,0.031778,0.168773,0.171602,0.280828,0.055089,0.278376,0.28328
"""moons""","""Tree""",12,2.25,0.999989,0.00005,0.999986,0.999992,0.999989,0.00005,0.999986,0.999992,0.9873,0.004414,0.987062,0.987539,0.987288,0.004435,0.987048,0.987527,0.1389,0.020565,0.137789,0.140011,0.236523,0.032432,0.234771,0.238276,0.349911,0.033049,0.348125,0.351697
"""moons""","""Tree""",12,2.5,0.999985,0.000056,0.999983,0.999988,0.999985,0.000056,0.999983,0.999988,0.987667,0.004172,0.987481,0.987852,0.987664,0.004184,0.987478,0.98785,0.139,0.022164,0.138013,0.139987,0.236258,0.03515,0.234693,0.237822,0.3472,0.04399,0.345242,0.349158


In [12]:
base = alt.Chart(
    df_stats.collect()
)

scale = alt.Scale(
    domain=["train acc_mean", "test acc_mean", "adv distance_mean"], 
    range=["blue", "red", "purple"]
)


dash_scale = alt.Scale(
    domain=["train acc_mean", "test acc_mean", "adv distance_mean"],
    range=[[2, 4], [1, 0], [8, 4]]  # [dash_length, gap_length]
)

# Chart for accuracy metrics (left y-axis)
acc_chart = base.mark_line(strokeWidth=2).transform_fold(
    fold=["train acc_mean", "test acc_mean"],
    as_=["variable", "value"]
).encode(
    x=alt.X("depth:Q").title("Depth"),
    y=alt.Y('value:Q').title("Accuracy").scale(zero=False),
    color=alt.Color('variable:N', scale=scale, title="Metric"),
    strokeDash=alt.StrokeDash('variable:N', scale=dash_scale)
)


# Chart for adversarial distance (right y-axis)
adv_chart = base.mark_line(strokeWidth=2).transform_fold(
    fold=["adv distance_mean"],
    as_=["variable", "value"]
).encode(
    x=alt.X("depth:Q").title("Depth"),
    y=alt.Y('value:Q').title("Adversarial Distance").axis(orient='right').scale(zero=False),
    color=alt.Color('variable:N', scale=scale, title="Metric"),
    strokeDash=alt.StrokeDash('variable:N', scale=dash_scale)

)

ci_band0 = base.mark_area(
    opacity=0.3,
    interpolate="basis",
).encode(
    x="depth:Q",
    y=alt.Y("train acc_ci_lb:Q").axis(title="Accuracy", orient="left"),
    y2=alt.Y2("train acc_ci_ub:Q"),
    color = alt.value("blue"),
)

ci_band1 = base.mark_area(
    opacity=0.3,
    interpolate="basis",
).encode(
    x="depth:Q",
    y=alt.Y("test acc_ci_lb:Q").axis(title="Accuracy", orient="left"),
    y2=alt.Y2("test acc_ci_ub:Q"),
    color = alt.value("red")
)

ci_band2 = base.mark_area(
    opacity=0.3,
    interpolate="basis",
).encode(
    x="depth:Q",
    y=alt.Y("adv distance_ci_lb:Q").axis(orient="right"),
    y2=alt.Y2("adv distance_ci_ub:Q"),
    color = alt.value("purple")
)

alt.layer(acc_chart + ci_band0 + ci_band1, adv_chart + ci_band2).resolve_scale(
    y='independent'
).facet(
    row="distribution",
    column="bias"
).resolve_scale(
    x="independent",
    y="independent"
)


alt.FacetChart(...)